In [1]:
import pandas                             as pd
import numpy                              as np
import matplotlib.pyplot                  as plt
import seaborn                            as sns


In [9]:
customers = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\customers.csv')
geography = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\geography.csv')
inventory = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\inventory.csv')
orders = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\orders.csv')
products = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\products.csv')
order_items = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\order_items.csv')
payments = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\payments.csv')
promotions = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\promotions.csv')
returns = pd.read_csv(filepath_or_buffer='datathon-2026-round-1/returns.csv')
reviews = pd.read_csv(filepath_or_buffer='datathon-2026-round-1/reviews.csv')
sales = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\sales.csv')
web_traffic = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\web_traffic.csv')


C:\Users\TAM\AppData\Local\Temp\ipykernel_23572\1129176885.py:6: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(filepath_or_buffer='datathon-2026-round-1\order_items.csv')


# **Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?**


In [ ]:

orders['order_date'] = pd.to_datetime(orders['order_date'])

orders = orders.sort_values(by=['customer_id', 'order_date'])

orders['inter_order_gap'] = orders.groupby('customer_id')['order_date'].diff().dt.days

gaps = orders['inter_order_gap'].dropna()

median_gap = gaps.median()

print(f"Trung vị số ngày giữa hai lần mua liên tiếp là: {median_gap} ngày")

Trung vị số ngày giữa hai lần mua liên tiếp là: 144.0 ngày


# **Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?**

In [21]:

products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

segment_margins = products.groupby('segment')['gross_margin'].mean()

best_segment = segment_margins.idxmax()
highest_value = segment_margins.max()

print("Tỷ suất lợi nhuận gộp trung bình theo từng phân khúc:")
print(segment_margins)
print(f"Phân khúc có tỷ suất lợi nhuận cao nhất là: {best_segment} ({highest_value:.4f})")

Tỷ suất lợi nhuận gộp trung bình theo từng phân khúc:
segment
Activewear     0.265600
All-weather    0.284176
Balanced       0.258038
Everyday       0.236343
Performance    0.263650
Premium        0.285377
Standard       0.313442
Trendy         0.240758
Name: gross_margin, dtype: float64
Phân khúc có tỷ suất lợi nhuận cao nhất là: Standard (0.3134)


# **Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?**

In [20]:

df_merged = pd.merge(returns, products, on='product_id', how='left')

#Lọc các sản phẩm thuộc danh mục 'Streetwear'
streetwear_returns = df_merged[df_merged['category'] == 'Streetwear']

reason_counts = streetwear_returns['return_reason'].value_counts()

most_common_reason = reason_counts.idxmax()
count_value = reason_counts.max()

print("Thống kê các lý do trả hàng cho danh mục Streetwear:")
print(reason_counts)
print(f"Lý do trả hàng xuất hiện nhiều nhất là: {most_common_reason} với {count_value} lượt.")


Thống kê các lý do trả hàng cho danh mục Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
Lý do trả hàng xuất hiện nhiều nhất là: wrong_size với 7626 lượt.


# **Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhấttrên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?**

In [19]:

avg_bounce_rates = web_traffic.groupby('traffic_source')['bounce_rate'].mean()

best_traffic_source = avg_bounce_rates.idxmin()
lowest_bounce_rate = avg_bounce_rates.min()

print("Tỷ lệ thoát trung bình theo từng nguồn truy cập:")
print(avg_bounce_rates)
print(f"Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: {best_traffic_source} ({lowest_bounce_rate:.4f})")

Tỷ lệ thoát trung bình theo từng nguồn truy cập:
traffic_source
direct            0.004511
email_campaign    0.004458
organic_search    0.004504
paid_search       0.004478
referral          0.004499
social_media      0.004476
Name: bounce_rate, dtype: float64
Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: email_campaign (0.0045)


# **Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?**

In [18]:
import pandas as pd


total_rows = len(order_items)


promo_count = order_items['promo_id'].notna().sum()

percentage = (promo_count / total_rows) * 100


print(f"Tỷ lệ phần trăm áp dụng khuyến mãi: {percentage:.2f}%")

Tỷ lệ phần trăm áp dụng khuyến mãi: 38.66%


# **Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)**

In [ ]:

customers = customers[customers['age_group'].notna()]

#Đếm số lượng đơn hàng của mỗi khách hàng từ file orders.csv
order_counts = orders.groupby('customer_id').size().reset_index(name='total_orders')

#Kết hợp bảng khách hàng với bảng đếm đơn hàng
#Sử dụng left join để giữ lại cả những khách hàng chưa từng mua hàng (nếu có trong customers)
df_merged = pd.merge(customers, order_counts, on='customer_id', how='left')

#Xử lý các khách hàng không có đơn hàng nào (NaN) thành 0
df_merged['total_orders'] = df_merged['total_orders'].fillna(0)

#Tính tổng số đơn và tổng số khách hàng cho mỗi nhóm tuổi
group_stats = df_merged.groupby('age_group').agg(
    total_orders_sum=('total_orders', 'sum'),
    total_customers=('customer_id', 'count')
)

#Tính chỉ số trung bình: Tổng số đơn / Số khách hàng
group_stats['avg_orders_per_customer'] = group_stats['total_orders_sum'] / group_stats['total_customers']

#Tìm nhóm tuổi có giá trị cao nhất
best_age_group = group_stats['avg_orders_per_customer'].idxmax()
max_val = group_stats['avg_orders_per_customer'].max()

print("Thống kê theo nhóm tuổi:")
print(group_stats['avg_orders_per_customer'].sort_values(ascending=False))
print(f"Nhóm tuổi có số đơn hàng trung bình cao nhất là: {best_age_group} ({max_val:.4f})")

Thống kê theo nhóm tuổi:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: avg_orders_per_customer, dtype: float64
Nhóm tuổi có số đơn hàng trung bình cao nhất là: 55+ (5.4069)


# **Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?**

In [29]:

df_order_payments = pd.merge(
    payments[['order_id', 'payment_value']], 
    orders[['order_id', 'zip']], 
    on='order_id'
)

# 3. Kết hợp với bảng geography qua cột 'zip' để xác định vùng (region)
df_final = pd.merge(
    df_order_payments, 
    geography[['zip', 'region']], 
    on='zip'
)

# 4. Tính tổng doanh thu (payment_value) theo từng vùng (region)
region_revenue = df_final.groupby('region')['payment_value'].sum().sort_values(ascending=False)

# Xuất kết quả
print("Tổng doanh thu theo từng vùng:")
print(region_revenue)
print(f"Vùng có doanh thu cao nhất là {region_revenue.idxmax()}")

Tổng doanh thu theo từng vùng:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64
Vùng có doanh thu cao nhất là East


# **Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?**

In [23]:

cancelled_orders = orders[orders['order_status'] == 'cancelled']

df_merged = pd.merge(
    cancelled_orders[['order_id']], 
    payments[['order_id', 'payment_method']], 
    on='order_id'
)

# 4. Đếm số lần xuất hiện của từng phương thức thanh toán
payment_method_counts = df_merged['payment_method'].value_counts()

# 5. Tìm phương thức thanh toán xuất hiện nhiều nhất
most_common_payment = payment_method_counts.idxmax()
max_count = payment_method_counts.max()

print("Thống kê phương thức thanh toán của các đơn hàng bị hủy:")
print(payment_method_counts)
print(f"Phương thức thanh toán được sử dụng nhiều nhất là {most_common_payment} ({max_count} lần).")

Thống kê phương thức thanh toán của các đơn hàng bị hủy:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64
Phương thức thanh toán được sử dụng nhiều nhất là credit_card (28452 lần).


# **Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?**

In [26]:

# Join order_items với products để lấy thông tin size
items_with_size = pd.merge(order_items[['product_id']], products[['product_id', 'size']], on='product_id', how='left')
order_counts = items_with_size['size'].value_counts()

# Join returns với products để lấy thông tin size
returns_with_size = pd.merge(returns[['product_id']], products[['product_id', 'size']], on='product_id')
return_counts = returns_with_size['size'].value_counts()

stats = pd.DataFrame({
    'total_orders': order_counts,
    'total_returns': return_counts
})

target_sizes = ['S', 'M', 'L', 'XL']
stats = stats.loc[target_sizes]

stats['return_rate'] = stats['total_returns'] / stats['total_orders']

highest_size = stats['return_rate'].idxmax()
highest_rate = stats['return_rate'].max()

print("Thống kê tỷ lệ trả hàng theo kích thước:")
print(stats.sort_values(by='return_rate', ascending=False))
print(f"Kích thước có tỷ lệ trả hàng cao nhất là {highest_size} ({highest_rate:.4f})")

Thống kê tỷ lệ trả hàng theo kích thước:
      total_orders  total_returns  return_rate
size                                          
S           172042           9723     0.056515
L           173174           9741     0.056250
M           176428           9820     0.055660
XL          193025          10655     0.055200
Kích thước có tỷ lệ trả hàng cao nhất là S (0.0565)


# **Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?**

In [28]:


installment_stats = payments.groupby('installments')['payment_value'].mean()

# 3. Sắp xếp để tìm kế hoạch có giá trị trung bình cao nhất
sorted_installments = installment_stats.sort_values(ascending=False)

# 4. Lấy kết quả cao nhất
best_plan = sorted_installments.idxmax()
highest_avg_value = sorted_installments.max()

print("Giá trị thanh toán trung bình theo từng số kỳ trả góp:")
print(sorted_installments.head(10)) # Hiển thị top 10 để đối chiếu
print(f"Đáp án Q10: Kế hoạch trả góp {best_plan} kỳ có giá trị trung bình cao nhất ({highest_avg_value:.2f})")

Giá trị thanh toán trung bình theo từng số kỳ trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
Đáp án Q10: Kế hoạch trả góp 6 kỳ có giá trị trung bình cao nhất (24446.65)
